In [8]:
#6
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 1. Load the dataset
df = pd.read_csv("ToyotaCorolla.csv")

# 2. Select only the requested predictors and the target variable (Price)
# [cite: 1440-1441]
features = ["Price", "Age_08_04", "KM", "Fuel_Type", "HP", "Automatic", 
            "Doors", "Quarterly_Tax", "Mfr_Guarantee", "Guarantee_Period", 
            "Airco", "Automatic_airco", "CD_Player", "Powered_Windows", 
            "Sport_Model", "Tow_Bar"]
df_subset = df[features].copy()

# 3. Convert Categorical variables to Dummies
# drop_first=True handles the N-1 rule automatically 
df_prepared = pd.get_dummies(df_subset, columns=['Fuel_Type'], drop_first=True)

# Separate X (predictors) and y (target)
X = df_prepared.drop('Price', axis=1)
y = df_prepared[['Price']] # Keeping as DataFrame for the scaler

# 4. Scale to 0-1 Range 
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_x.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel() # .ravel() converts to 1D array for the model

# 5. Partition Data: 80% Training, 20% Validation [cite: 1471]
X_train, X_valid, y_train, y_valid = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# 6. Define the three requested architectures [cite: 1468-1470]
architectures = {
    "Single Layer (2 Nodes)": (2,),
    "Single Layer (5 Nodes)": (5,),
    "Two Layers (5 Nodes each)": (5, 5)
}

# 7. Train and evaluate each model
for name, hidden_layers in architectures.items():
    # max_iter increased slightly to ensure convergence
    nn = MLPRegressor(hidden_layer_sizes=hidden_layers, activation='logistic', solver='lbfgs', max_iter=1000, random_state=42)
    nn.fit(X_train, y_train)
    
    # Predictions
    pred_train = nn.predict(X_train)
    pred_valid = nn.predict(X_valid)
    
    # Calculate RMSE
    rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
    rmse_valid = np.sqrt(mean_squared_error(y_valid, pred_valid))
    
    print(f"--- {name} ---")
    print(f"Training RMSE:   {rmse_train:.5f}")
    print(f"Validation RMSE: {rmse_valid:.5f}\n")

--- Single Layer (2 Nodes) ---
Training RMSE:   0.03991
Validation RMSE: 0.03516

--- Single Layer (5 Nodes) ---
Training RMSE:   0.03948
Validation RMSE: 0.03506

--- Two Layers (5 Nodes each) ---
Training RMSE:   0.03961
Validation RMSE: 0.03534



In [9]:
#5
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Load data
df = pd.read_csv("accidentsFull.csv")

# 2. Create the target dummy variable (INJURY: yes/no) [cite: 734]
df['INJURY'] = df['MAX_SEV_IR'].apply(lambda x: 'yes' if x > 0 else 'no')

# 3. Select relevant categorical predictors
features = ["HOUR_I_R", "ALIGN_I", "WRK_ZONE", "WKDY_I_R", "INT_HWY", 
            "LGTCON_I_R", "PROFIL_I_R", "SPD_LIM", "SUR_COND", 
            "TRAF_CON_R", "TRAF_WAY", "WEATHER_R"]

X = df[features]
y = df['INJURY']

# 4. Partition data (60% Training, 40% Validation) [cite: 1242]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.4, random_state=42)

# 5. Train the Naive Bayes Classifier
# MultinomialNB is perfect for categorical/frequency data
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# 6. Make Predictions on Validation Set
y_pred = nb_model.predict(X_valid)

# 7. Output Confusion Matrix and Accuracy
print("Confusion Matrix:")
print(confusion_matrix(y_valid, y_pred))

accuracy = accuracy_score(y_valid, y_pred)
print(f"\nOverall Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Overall Validation Error: {(1 - accuracy) * 100:.2f}%")

Confusion Matrix:
[[3078 5195]
 [2859 5742]]

Overall Validation Accuracy: 52.27%
Overall Validation Error: 47.73%


In [11]:
pip install mlxtend

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------------------------------ --------- 1.0/1.4 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 4.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:
#7
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Load the data
df = pd.read_csv("CourseTopics.csv")

# 2. Generate Frequent Itemsets
# min_support=0.05 means the combination must appear in at least 5% of all purchases
frequent_itemsets = apriori(df, min_support=0.05, use_colnames=True)

# 3. Generate Association Rules
# We use Lift as our metric, looking for rules with a lift greater than 1.2
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

# 4. Sort the rules by the highest Lift to find the strongest patterns
rules_sorted = rules.sort_values(by='lift', ascending=False)

# Display the top 5 most actionable course combinations
print(rules_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

  antecedents consequents   support  confidence      lift
0    (Survey)  (Cat Data)  0.063014    0.338235  1.624420
1  (Cat Data)    (Survey)  0.063014    0.302632  1.624420
6       (DOE)        (SW)  0.057534    0.333333  1.502058
7        (SW)       (DOE)  0.057534    0.259259  1.502058
5  (Cat Data)        (SW)  0.063014    0.302632  1.363710


c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [2]:
#Practical 5: Automobile Accidents (Naive Bayes)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Load data
df = pd.read_csv("accidentsFull.csv")

# 2. Create the target dummy variable (INJURY: yes/no)
df['INJURY'] = df['MAX_SEV_IR'].apply(lambda x: 'yes' if x > 0 else 'no')

# 3. Select relevant categorical predictors 
features = ["HOUR_I_R", "ALIGN_I", "WRK_ZONE", "WKDY_I_R", "INT_HWY", 
            "LGTCON_I_R", "PROFIL_I_R", "SPD_LIM", "SUR_COND", 
            "TRAF_CON_R", "TRAF_WAY", "WEATHER_R"]

X = df[features]
y = df['INJURY']

# 4. Partition data (60% Training, 40% Validation)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.4, random_state=42)

# 5. Train the Naive Bayes Classifier
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# 6. Make Predictions and Output Results
y_pred = nb_model.predict(X_valid)

print("Confusion Matrix:")
print(confusion_matrix(y_valid, y_pred))

accuracy = accuracy_score(y_valid, y_pred)
print(f"\nOverall Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Overall Validation Error: {(1 - accuracy) * 100:.2f}%")

Confusion Matrix:
[[3078 5195]
 [2859 5742]]

Overall Validation Accuracy: 52.27%
Overall Validation Error: 47.73%


In [3]:
#Practical 6: Car Sales (Neural Networks)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 1. Load data and select features
df = pd.read_csv("ToyotaCorolla.csv")
features = ["Price", "Age_08_04", "KM", "Fuel_Type", "HP", "Automatic", 
            "Doors", "Quarterly_Tax", "Mfr_Guarantee", "Guarantee_Period", 
            "Airco", "Automatic_airco", "CD_Player", "Powered_Windows", 
            "Sport_Model", "Tow_Bar"]
df_subset = df[features].copy()

# 2. Convert Categorical variables to Dummies & Separate X/y
df_prepared = pd.get_dummies(df_subset, columns=['Fuel_Type'], drop_first=True)
X = df_prepared.drop('Price', axis=1)
y = df_prepared[['Price']] 

# 3. Scale to 0-1 Range (Crucial for Neural Nets)
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_x.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel() 

# 4. Partition Data (80% Training, 20% Validation)
X_train, X_valid, y_train, y_valid = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# 5. Train and evaluate the three architectures
architectures = {"Single Layer (2 Nodes)": (2,), "Single Layer (5 Nodes)": (5,), "Two Layers (5 Nodes each)": (5, 5)}

for name, hidden_layers in architectures.items():
    nn = MLPRegressor(hidden_layer_sizes=hidden_layers, activation='logistic', solver='lbfgs', max_iter=1000, random_state=42)
    nn.fit(X_train, y_train)
    
    rmse_train = np.sqrt(mean_squared_error(y_train, nn.predict(X_train)))
    rmse_valid = np.sqrt(mean_squared_error(y_valid, nn.predict(X_valid)))
    
    print(f"--- {name} ---")
    print(f"Training RMSE:   {rmse_train:.5f}")
    print(f"Validation RMSE: {rmse_valid:.5f}\n")

--- Single Layer (2 Nodes) ---
Training RMSE:   0.03991
Validation RMSE: 0.03516

--- Single Layer (5 Nodes) ---
Training RMSE:   0.03948
Validation RMSE: 0.03506

--- Two Layers (5 Nodes each) ---
Training RMSE:   0.03961
Validation RMSE: 0.03534



In [4]:
#Practical 7: Online Statistics Courses (Association Rules)
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Load the data
df = pd.read_csv("CourseTopics.csv")

# 2. Generate Frequent Itemsets
# min_support=0.05 means the combination must appear in at least 5% of all purchases
frequent_itemsets = apriori(df, min_support=0.05, use_colnames=True)

# 3. Generate Association Rules
# We use Lift as our metric, looking for rules with a lift greater than 1.2
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

# 4. Sort the rules by the highest Lift to find the strongest patterns
rules_sorted = rules.sort_values(by='lift', ascending=False)

# Display the top rules for interpretation
print(rules_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

  antecedents consequents   support  confidence      lift
0    (Survey)  (Cat Data)  0.063014    0.338235  1.624420
1  (Cat Data)    (Survey)  0.063014    0.302632  1.624420
6       (DOE)        (SW)  0.057534    0.333333  1.502058
7        (SW)       (DOE)  0.057534    0.259259  1.502058
5  (Cat Data)        (SW)  0.063014    0.302632  1.363710


c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
